# 01 Revisión práctica de Machine Learning

## Propósito

Repasar los fundamentos de Machine Learning. Compararemos métodos sobre los mismos datasets y explicaremos sus resultados.

Esta práctica corresponde a una revisión de contenidos previos de Machine Learning. No requiere conocimientos de Recuperación de Información.

## Objetivos

Al completar el notebook podrás:

- Distinguir clasificación, regresión, clustering, recomendación y asociación.
- Preparar datos y separar entrenamiento, validación y prueba.
- Comparar modelos mediante métricas apropiadas.
- Explicar los efectos del escalamiento, los hiperparámetros y la regularización.
- Interpretar softmax y las salidas de un MLP.
- Identificar limitaciones y errores de los modelos.

## Uso de IA Generativa

Puedes usar IA para generar, corregir y explicar código. Debes comprobar que el código corresponde al experimento solicitado y poder explicar las decisiones y los resultados.

Antes de solicitar una implementación, registra tu predicción sobre el experimento.

## Organización del trabajo

Después de cada bloque de instrucciones, añade las celdas de código y Markdown necesarias.

Cada experimento debe incluir:

1. Una predicción inicial.
2. La implementación y sus resultados.
3. Una interpretación basada en los conceptos de ML.
4. Una respuesta a las preguntas planteadas.

Las tablas y figuras deben generarse a partir de los resultados obtenidos.

# Soporte (S0–S6)

Esta sección agrupa, **fuera de los experimentos**, todo lo que los tres experimentos comparten: los `import`, la semilla única, las funciones de lógica y sus comprobaciones. **No es un experimento**: sus celdas no siguen la regla de tres celdas «predicción → implementación → interpretación», que sí rige para E1, E2 y E3.

## Cómo está organizado

| Celda | Contenido |
|---|---|
| **S1** | Imports del stack fijado, semilla única `SEED`, límite `MAX_CONFIGS`, orden y etiquetas de los modelos |
| **S2** | Datos y particiones: `make_splits`, `split_summary`, `sample_examples` |
| **S3** | Fábricas de modelos: `build_classifier`, `build_regressor` |
| **S4** | Protocolo: `fit_capturing_warnings`, `classification_metrics`, `regression_metrics`, `score_classifier`, `score_regressor`, `train_and_score`, `select_config`, `FrozenModel`/`freeze`, `evaluate_once`, `alpha_sweep`, `pick_best_alpha` |
| **S5** | Análisis: `make_table`, `format_config`, `format_warnings`, `simplicity_score`, `indistinguishable_cls`, `indistinguishable_reg`, `rank_models`, `never_predicted_classes`, `is_degenerate_cls`, `is_degenerate_reg`, `top_abs_errors`, `pick_two_errors`, `result_digest` |
| **S6** | Un `assert` como mínimo por cada función de S2–S5, con datos sintéticos (sin pytest) |

## Reglas que se cumplen en todo el notebook

- Las funciones **devuelven valores**: no imprimen, no muestran tablas y no dibujan. Mostrar y graficar ocurre solo en las celdas de implementación de cada experimento.
- Una **configuración** es una combinación distinta de hiperparámetros (incluidos `max_iter` y `solver`); la semilla no cuenta y se inyecta desde `SEED`.
- **Congelar** es fijar la configuración y el modelo ajustado solo con entrenamiento; no se reentrena con validación y la partición de prueba se evalúa una sola vez por modelo.
- Los avisos de scikit-learn se **capturan y se muestran en tablas**; nunca se silencian.
- Solo cuenta como válida la ejecución completa **desde cero** (*Restart & Run All*).

In [1]:
# [S1] Imports, single seed and protocol constants
import hashlib
import warnings
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.datasets import load_diabetes, load_iris
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                             r2_score, root_mean_squared_error)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# Single source of randomness: every random_state and every RNG below receives this value.
SEED = 42

# Maximum number of configurations per model, the initial one included.
# The alpha sweep of Ridge and Lasso is the only declared exception.
MAX_CONFIGS = 4

# Model order is fixed by the statement (classification: block 2, regression: block 3).
CLF_KEYS = ["gaussian_nb", "logistic_regression", "linear_svm", "decision_tree",
            "random_forest", "gradient_boosting", "mlp"]
CLF_LABELS = {
    "gaussian_nb": "Gaussian Naive Bayes",
    "logistic_regression": "Regresión logística",
    "linear_svm": "SVM lineal",
    "decision_tree": "Árbol de decisión",
    "random_forest": "Random Forest",
    "gradient_boosting": "Gradient Boosting",
    "mlp": "MLP",
}
REG_KEYS = ["linear_regression", "ridge", "lasso", "knn", "decision_tree",
            "random_forest", "gradient_boosting", "mlp"]
REG_LABELS = {
    "linear_regression": "Regresión lineal",
    "ridge": "Ridge",
    "lasso": "Lasso",
    "knn": "kNN",
    "decision_tree": "Árbol de regresión",
    "random_forest": "Random Forest",
    "gradient_boosting": "Gradient Boosting",
    "mlp": "MLP",
}
PARTITION_LABELS = {"train": "entrenamiento", "val": "validación", "test": "prueba"}

# Table display options (presentation only).
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)

In [2]:
# [S2] Data and partition logic (returns values; nothing is shown or drawn here)
def make_splits(y, stratify, seed):
    """Return sorted index arrays for a 60/20/20 train/val/test partition (two chained splits)."""
    y = np.asarray(y)
    idx = np.arange(len(y))
    rest, test = train_test_split(idx, test_size=0.20, random_state=seed,
                                  stratify=y if stratify else None)
    train, val = train_test_split(rest, test_size=0.25, random_state=seed,
                                  stratify=y[rest] if stratify else None)
    return {"train": np.sort(train), "val": np.sort(val), "test": np.sort(test)}


def split_summary(y, splits, class_names):
    """Size and per-class counts of each partition, as a DataFrame with Spanish headers."""
    y = np.asarray(y)
    rows = []
    for name in ("train", "val", "test"):
        part = y[splits[name]]
        row = {"Partición": PARTITION_LABELS[name], "Tamaño": int(len(part))}
        for code, label in enumerate(class_names):
            row[label] = int(np.sum(part == code))
        rows.append(row)
    return pd.DataFrame(rows)


def sample_examples(y, n, seed):
    """Pick n distinct indices at random, guaranteeing that every class appears at least once."""
    y = np.asarray(y)
    classes = np.unique(y)
    if n < len(classes):
        raise ValueError("n must be at least the number of classes")
    rng = np.random.default_rng(seed)
    chosen = [int(rng.choice(np.flatnonzero(y == c))) for c in classes]
    remaining = np.setdiff1d(np.arange(len(y)), chosen)
    extra = rng.choice(remaining, size=n - len(chosen), replace=False)
    return np.sort(np.concatenate([np.array(chosen, dtype=int), extra.astype(int)]))

In [3]:
# [S3] Model factories (the seed is injected here and is never part of a config)
def build_classifier(key, config, seed, scaled):
    """Return an unfitted Pipeline; the scaler step exists only when scaled is True."""
    if "random_state" in config:
        raise ValueError("the seed is injected by the factory, not written in the config")
    if key == "gaussian_nb":
        model = GaussianNB(**config)
    elif key == "logistic_regression":
        model = LogisticRegression(**config, random_state=seed)
    elif key == "linear_svm":
        model = LinearSVC(**config, random_state=seed)
    elif key == "decision_tree":
        model = DecisionTreeClassifier(**config, random_state=seed)
    elif key == "random_forest":
        model = RandomForestClassifier(**config, random_state=seed)
    elif key == "gradient_boosting":
        model = GradientBoostingClassifier(**config, random_state=seed)
    elif key == "mlp":
        model = MLPClassifier(**config, random_state=seed)
    else:
        raise ValueError(f"unknown classifier key: {key}")
    steps = [("scaler", StandardScaler())] if scaled else []
    return Pipeline(steps + [("model", model)])


# Regressors that need standardized inputs (distance-based, penalized or gradient-trained).
SCALED_REGRESSORS = ("ridge", "lasso", "knn", "mlp")


def build_regressor(key, config, seed):
    """Return an unfitted regressor; only the MLP wraps the target in a train-fitted scaler."""
    if "random_state" in config:
        raise ValueError("the seed is injected by the factory, not written in the config")
    if key == "linear_regression":
        model = LinearRegression(**config)
    elif key == "ridge":
        model = Ridge(**config, random_state=seed)
    elif key == "lasso":
        model = Lasso(**config, random_state=seed)
    elif key == "knn":
        model = KNeighborsRegressor(**config)
    elif key == "decision_tree":
        model = DecisionTreeRegressor(**config, random_state=seed)
    elif key == "random_forest":
        model = RandomForestRegressor(**config, random_state=seed)
    elif key == "gradient_boosting":
        model = GradientBoostingRegressor(**config, random_state=seed)
    elif key == "mlp":
        model = MLPRegressor(**config, random_state=seed)
    else:
        raise ValueError(f"unknown regressor key: {key}")
    steps = [("scaler", StandardScaler())] if key in SCALED_REGRESSORS else []
    pipe = Pipeline(steps + [("model", model)])
    if key == "mlp":
        # The target is standardized with train statistics and inverted before any metric.
        return TransformedTargetRegressor(regressor=pipe, transformer=StandardScaler())
    return pipe

In [4]:
# [S4] Protocol logic: fit, score, select, freeze, single test evaluation and alpha sweep
def fit_capturing_warnings(est, X, y):
    """Fit est and return (est, warnings). Every warning is kept; none is filtered out."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        est.fit(X, y)
    return est, [{"category": w.category.__name__, "message": str(w.message)} for w in caught]


def classification_metrics(y_true, y_pred):
    """Accuracy, macro F1 and micro F1 of a prediction vector."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "f1_micro": float(f1_score(y_true, y_pred, average="micro")),
    }


def regression_metrics(y_true, y_pred):
    """MAE, RMSE and R2 of a prediction vector (original target scale)."""
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(root_mean_squared_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def score_classifier(est, X_eval, y_eval):
    """Predict with a fitted estimator; return (y_pred, metrics, warnings)."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        y_pred = est.predict(X_eval)
        metrics = classification_metrics(y_eval, y_pred)
    return y_pred, metrics, [{"category": w.category.__name__, "message": str(w.message)}
                             for w in caught]


def score_regressor(est, X_eval, y_eval):
    """Predict with a fitted estimator; return (y_pred, metrics, warnings)."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        y_pred = est.predict(X_eval)
        metrics = regression_metrics(y_eval, y_pred)
    return y_pred, metrics, [{"category": w.category.__name__, "message": str(w.message)}
                             for w in caught]


def train_and_score(est, X_fit, y_fit, X_eval, y_eval, scorer):
    """Fit on the fitting data only, score on the evaluation data, keep all warnings."""
    est, fit_warnings = fit_capturing_warnings(est, X_fit, y_fit)
    y_pred, metrics, eval_warnings = scorer(est, X_eval, y_eval)
    return {"estimator": est, "y_pred": y_pred, **metrics,
            "warnings_fit": fit_warnings, "warnings_eval": eval_warnings}


def select_config(records, primary, support, higher_is_better):
    """Pick the best record: primary metric, then support metric, then the one tried first."""
    sign = 1.0 if higher_is_better else -1.0
    return max(records, key=lambda r: (round(sign * r[primary], 12),
                                       round(sign * r[support], 12),
                                       -r["order"]))


class FrozenModel(NamedTuple):
    """Immutable pair (configuration, estimator fitted on train only)."""
    model: str
    config: dict
    estimator: object
    n_configs: int


def freeze(key, record, n_configs):
    """Freeze a chosen record without refitting: the estimator was fitted on train only."""
    return FrozenModel(model=key, config=dict(record["config"]),
                       estimator=record["estimator"], n_configs=int(n_configs))


def evaluate_once(frozen, X_eval, y_eval, ledger, scorer):
    """Score a frozen model on the held-out partition; fails if it was already evaluated."""
    if frozen.model in ledger:
        raise RuntimeError(f"{frozen.model} was already evaluated on the held-out partition")
    ledger.add(frozen.model)
    y_pred, metrics, caught = scorer(frozen.estimator, X_eval, y_eval)
    return {"y_pred": y_pred, **metrics, "warnings_eval": caught}


def alpha_sweep(key, base_config, alphas, data, seed):
    """One fitted pipeline per alpha (fit on data['X_fit']), scored on fit and eval data."""
    rows = []
    for alpha in alphas:
        config = {**base_config, "alpha": float(alpha)}
        est = build_regressor(key, config, seed)
        est, fit_warnings = fit_capturing_warnings(est, data["X_fit"], data["y_fit"])
        _, m_fit, _ = score_regressor(est, data["X_fit"], data["y_fit"])
        _, m_eval, eval_warnings = score_regressor(est, data["X_eval"], data["y_eval"])
        coef = est.named_steps["model"].coef_
        rows.append({
            "model": key, "alpha": float(alpha), "config": config,
            "l2_norm": float(np.linalg.norm(coef)), "n_zero": int(np.sum(coef == 0.0)),
            "rmse_train": m_fit["rmse"], "rmse_val": m_eval["rmse"], "mae_val": m_eval["mae"],
            "coef": np.array(coef, copy=True),
            "warnings_fit": fit_warnings, "warnings_eval": eval_warnings, "estimator": est,
        })
    return rows


def pick_best_alpha(rows):
    """Lowest validation RMSE; an exact tie goes to the smaller alpha. Flags the grid border."""
    best = min(rows, key=lambda r: (round(r["rmse_val"], 12), r["alpha"]))
    alphas = [r["alpha"] for r in rows]
    return {"alpha": best["alpha"],
            "at_border": best["alpha"] in (min(alphas), max(alphas)),
            "row": best}

In [5]:
# [S5] Analysis logic: comparisons, degenerate results, errors and fingerprints
def make_table(rows, columns):
    """DataFrame with exactly the given columns and a 1-based index (presentation is up to the caller)."""
    table = pd.DataFrame(rows, columns=columns)
    table.index = pd.RangeIndex(1, len(table) + 1)
    return table


def format_config(config):
    """Text with only the hyperparameters explicitly written in the configuration."""
    if not config:
        return "sin hiperparámetros fijados"
    parts = []
    for name, value in config.items():
        shown = float(f"{value:.6g}") if isinstance(value, float) else value
        parts.append(f"{name}={shown}")
    return ", ".join(parts)


def format_warnings(warns):
    """Compact text of captured warnings (category and count), or a dash when there are none."""
    if not warns:
        return "—"
    counts = {}
    for item in warns:
        counts[item["category"]] = counts.get(item["category"], 0) + 1
    return "; ".join(f"{name} ×{count}" for name, count in counts.items())


def simplicity_score(frozen):
    """Explicitly fixed hyperparameters plus configurations tried (lower = simpler to program)."""
    return len(frozen.config) + frozen.n_configs


def indistinguishable_cls(errors, best, tol=1):
    """Models (other than best) with at most tol more test errors than the best one."""
    return [m for m, e in errors.items() if m != best and e - errors[best] <= tol]


def indistinguishable_reg(rmse, best, tol=0.05):
    """Models (other than best) within tol (relative to the best RMSE) of the best RMSE."""
    return [m for m, v in rmse.items()
            if m != best and (v - rmse[best]) / rmse[best] <= tol + 1e-12]


def rank_models(values, higher_is_better):
    """Competition ranking (1 = best); ties share the same position."""
    sign = -1.0 if higher_is_better else 1.0
    keyed = {m: round(sign * float(v), 12) for m, v in values.items()}
    return {m: 1 + sum(1 for other in keyed.values() if other < keyed[m]) for m in keyed}


def never_predicted_classes(y_pred, classes):
    """Classes (in the given order) that the prediction vector never contains."""
    seen = set(np.asarray(y_pred).tolist())
    return [int(c) for c in classes if int(c) not in seen]


def is_degenerate_cls(y_pred, classes):
    """True when the classifier never predicts at least one class."""
    return len(never_predicted_classes(y_pred, classes)) > 0


def is_degenerate_reg(est):
    """True when a linear model has every coefficient exactly zero (a constant predictor)."""
    model = est.named_steps["model"] if hasattr(est, "named_steps") else None
    coef = getattr(model, "coef_", None)
    return coef is not None and bool(np.all(np.asarray(coef) == 0.0))


def top_abs_errors(y_true, y_pred, k=2):
    """Positions of the k largest absolute errors (ties go to the smaller position)."""
    err = np.abs(np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float))
    order = sorted(range(len(err)), key=lambda i: (-err[i], i))
    return np.array(order[:k], dtype=int)


def pick_two_errors(pred_main, pred_extra, y_main, y_extra, model_order, idx_main, idx_extra):
    """Up to two distinct misclassified instances, taken from the main (held-out) partition
    first and completed, if needed, from the extra (validation) partition with a label."""
    chosen = []
    wrong_main = {m: np.asarray(pred_main[m]) != np.asarray(y_main) for m in model_order}
    fails = np.sum([wrong_main[m] for m in model_order], axis=0)
    candidates = sorted((i for i in range(len(y_main)) if fails[i] > 0),
                        key=lambda i: (-int(fails[i]), int(idx_main[i])))
    for i in candidates[:2]:
        first = next(m for m in model_order if wrong_main[m][i])
        chosen.append({"index": int(idx_main[i]), "position": int(i), "partition": "prueba",
                       "model": first, "also_failed": int(fails[i]) - 1, "note": ""})
    if len(chosen) < 2:
        wrong_extra = {m: np.asarray(pred_extra[m]) != np.asarray(y_extra) for m in model_order}
        fails_extra = np.sum([wrong_extra[m] for m in model_order], axis=0)
        by_errors = sorted(model_order,
                           key=lambda m: (-int(wrong_extra[m].sum()), model_order.index(m)))
        seen = {c["index"] for c in chosen}
        for m in by_errors:
            for i in sorted(np.flatnonzero(wrong_extra[m]), key=lambda j: int(idx_extra[j])):
                if len(chosen) >= 2:
                    break
                if int(idx_extra[i]) in seen:
                    continue
                seen.add(int(idx_extra[i]))
                chosen.append({"index": int(idx_extra[i]), "position": int(i),
                               "partition": "validación", "model": m,
                               "also_failed": int(fails_extra[i]) - 1,
                               "note": "posible sesgo optimista: la validación se usó para decidir"})
    return chosen


def result_digest(*arrays):
    """Short SHA-256 of the arrays rounded to 6 decimals (shape included)."""
    digest = hashlib.sha256()
    for array in arrays:
        values = np.asarray(array, dtype=float)
        digest.update(str(values.shape).encode())
        digest.update((np.round(values, 6) + 0.0).tobytes())
    return digest.hexdigest()[:16]

In [6]:
# [S6] Checks of every logic function of S2-S5 (plain asserts, synthetic data only)

# ---- S2: make_splits, split_summary, sample_examples ----
_y3 = np.array([0] * 50 + [1] * 50 + [2] * 50)
_sp = make_splits(_y3, stratify=True, seed=SEED)
assert [len(_sp[k]) for k in ("train", "val", "test")] == [90, 30, 30], "make_splits: sizes must be 90/30/30"
_union = np.concatenate([_sp["train"], _sp["val"], _sp["test"]])
assert sorted(_union.tolist()) == list(range(150)), "make_splits: partitions must be disjoint and complete"
for _part, _per_class in (("train", 30), ("val", 10), ("test", 10)):
    assert np.bincount(_y3[_sp[_part]]).tolist() == [_per_class] * 3, "make_splits: stratification broken"
_sp_again = make_splits(_y3, stratify=True, seed=SEED)
assert all(np.array_equal(_sp[k], _sp_again[k]) for k in _sp), "make_splits: same seed, same indices"
_sp_other = make_splits(_y3, stratify=True, seed=SEED + 1)
assert not np.array_equal(_sp["test"], _sp_other["test"]), "make_splits: another seed must change the split"
_sp_reg = make_splits(np.zeros(442), stratify=False, seed=SEED)
assert [len(_sp_reg[k]) for k in ("train", "val", "test")] == [264, 89, 89], "make_splits: 442 rows -> 264/89/89"

_summary = split_summary(_y3, _sp, ["a", "b", "c"])
assert _summary["Tamaño"].tolist() == [90, 30, 30], "split_summary: sizes"
assert _summary[["a", "b", "c"]].to_numpy().tolist() == [[30, 30, 30], [10, 10, 10], [10, 10, 10]], \
    "split_summary: per-class counts"

_ex = sample_examples(_y3, 5, SEED)
assert len(set(_ex.tolist())) == 5, "sample_examples: indices must be distinct"
assert set(_y3[_ex].tolist()) == {0, 1, 2}, "sample_examples: all classes must appear"
assert np.array_equal(_ex, sample_examples(_y3, 5, SEED)), "sample_examples: must be deterministic"

# ---- S3: build_classifier, build_regressor ----
assert len(CLF_KEYS) == 7 and len(REG_KEYS) == 8, "model lists must have 7 classifiers and 8 regressors"
for _scaled in (True, False):
    for _key in CLF_KEYS:
        _est = build_classifier(_key, {}, SEED, scaled=_scaled)
        _step_names = [name for name, _ in _est.steps]
        assert _step_names == (["scaler", "model"] if _scaled else ["model"]), "build_classifier: scaler step"
        if _scaled:
            assert isinstance(_est.named_steps["scaler"], StandardScaler), "build_classifier: StandardScaler"
        _params = _est.named_steps["model"].get_params()
        assert _params.get("random_state", SEED) == SEED, "build_classifier: random_state must be SEED"
assert isinstance(build_classifier("linear_svm", {}, SEED, True).named_steps["model"], LinearSVC)
assert build_classifier("mlp", {}, SEED, True).named_steps["model"].early_stopping is False, \
    "MLP must not use an internal validation split"
assert build_classifier("gradient_boosting", {}, SEED, True).named_steps["model"].n_iter_no_change is None, \
    "Gradient Boosting must not use an internal validation split"

for _key in REG_KEYS:
    _est = build_regressor(_key, {}, SEED)
    _core = _est.regressor if isinstance(_est, TransformedTargetRegressor) else _est
    assert ("scaler" in _core.named_steps) == (_key in ("ridge", "lasso", "knn", "mlp")), "build_regressor: scaler"
    assert _core.named_steps["model"].get_params().get("random_state", SEED) == SEED, "build_regressor: seed"
assert [isinstance(build_regressor(k, {}, SEED), TransformedTargetRegressor) for k in REG_KEYS] == \
    [k == "mlp" for k in REG_KEYS], "only the MLP wraps the target"
_mlp_core = build_regressor("mlp", {}, SEED).regressor.named_steps["model"]
assert _mlp_core.early_stopping is False, "MLP regressor must not use an internal validation split"
assert build_regressor("gradient_boosting", {}, SEED).named_steps["model"].n_iter_no_change is None

# ---- S4: fit_capturing_warnings, metrics, score_*, train_and_score ----
_rng = np.random.default_rng(SEED)
_Xc = _rng.normal(size=(90, 4))
_yc = np.digitize(_Xc[:, 0] + 0.3 * _Xc[:, 1], [-0.5, 0.5])
_lr, _lr_warnings = fit_capturing_warnings(LogisticRegression(max_iter=1), _Xc, _yc)
assert hasattr(_lr, "coef_"), "fit_capturing_warnings: estimator must come back fitted"
assert any(w["category"] == "ConvergenceWarning" for w in _lr_warnings), "ConvergenceWarning must be captured"

_m = classification_metrics([0, 0, 1, 1, 2, 2], [0, 0, 1, 2, 2, 2])
assert abs(_m["f1_macro"] - (1 + 2 / 3 + 4 / 5) / 3) < 1e-12, "classification_metrics: macro F1"
assert abs(_m["f1_micro"] - 5 / 6) < 1e-12 and abs(_m["accuracy"] - 5 / 6) < 1e-12, "micro F1 equals accuracy"
_r = regression_metrics([1, 2, 3], [1, 2, 5])
assert abs(_r["mae"] - 2 / 3) < 1e-12 and abs(_r["rmse"] - np.sqrt(4 / 3)) < 1e-12 and abs(_r["r2"] + 1) < 1e-12, \
    "regression_metrics: MAE, RMSE and R2 of a known case"

_pred, _met, _score_warnings = score_classifier(_lr, _Xc, _yc)
assert _met == classification_metrics(_yc, _pred) and isinstance(_score_warnings, list), "score_classifier"
_Xr = _rng.normal(size=(60, 3))
_yr = _Xr @ np.array([2.0, -1.0, 0.5]) + 10
_lin, _ = fit_capturing_warnings(build_regressor("linear_regression", {}, SEED), _Xr, _yr)
_pred_r, _met_r, _ = score_regressor(_lin, _Xr, _yr)
assert _met_r == regression_metrics(_yr, _pred_r) and _met_r["r2"] > 0.999, "score_regressor"

_res = train_and_score(build_classifier("gaussian_nb", {}, SEED, True), _Xc, _yc, _Xc, _yc, score_classifier)
assert set(_res) >= {"estimator", "y_pred", "accuracy", "f1_macro", "warnings_fit", "warnings_eval"}, \
    "train_and_score: result keys"

# ---- S4: select_config ----
_recs = [{"order": 0, "f1": 0.9, "acc": 0.8}, {"order": 1, "f1": 0.9, "acc": 0.9}, {"order": 2, "f1": 0.8, "acc": 1.0}]
assert select_config(_recs, "f1", "acc", True)["order"] == 1, "select_config: tie goes to the support metric"
_recs_tie = [{"order": 0, "f1": 0.9, "acc": 0.9}, {"order": 1, "f1": 0.9, "acc": 0.9}]
assert select_config(_recs_tie, "f1", "acc", True)["order"] == 0, "select_config: persistent tie, first tried"
_recs_reg = [{"order": 0, "rmse": 5.0, "mae": 4.0}, {"order": 1, "rmse": 3.0, "mae": 4.5},
             {"order": 2, "rmse": 3.0, "mae": 4.0}]
assert select_config(_recs_reg, "rmse", "mae", False)["order"] == 2, "select_config: lower is better"

# ---- S4: freeze, evaluate_once ----
_nb, _ = fit_capturing_warnings(build_classifier("gaussian_nb", {}, SEED, True), _Xc, _yc)
_frozen = freeze("gaussian_nb", {"order": 0, "config": {"var_smoothing": 1e-9}, "estimator": _nb}, 3)
assert _frozen.model == "gaussian_nb" and _frozen.n_configs == 3 and _frozen.estimator is _nb, "freeze fields"
try:
    _frozen.model = "other"
    raise AssertionError("FrozenModel must be immutable")
except AttributeError:
    pass
_ledger = set()
_first = evaluate_once(_frozen, _Xc, _yc, _ledger, score_classifier)
assert _ledger == {"gaussian_nb"} and np.array_equal(_first["y_pred"], _nb.predict(_Xc)), "evaluate_once"
try:
    evaluate_once(_frozen, _Xc, _yc, _ledger, score_classifier)
    raise AssertionError("a second evaluation of the same model must fail")
except RuntimeError:
    pass

# ---- S4: alpha_sweep, pick_best_alpha ----
_Xa = _rng.normal(size=(120, 5))
_ya = _Xa @ np.array([3.0, -2.0, 0.0, 0.0, 1.0]) + _rng.normal(scale=0.5, size=120)
_data = {"X_fit": _Xa[:80], "y_fit": _ya[:80], "X_eval": _Xa[80:], "y_eval": _ya[80:]}
_ridge_rows = alpha_sweep("ridge", {}, np.logspace(-2, 2, 6), _data, SEED)
_norms = [row["l2_norm"] for row in _ridge_rows]
assert all(_norms[i] >= _norms[i + 1] for i in range(len(_norms) - 1)), "Ridge L2 norm must not grow with alpha"
_Xs = StandardScaler().fit_transform(_data["X_fit"])
_alpha_max = np.max(np.abs(_Xs.T @ (_data["y_fit"] - _data["y_fit"].mean()))) / len(_Xs)
_lasso_rows = alpha_sweep("lasso", {"max_iter": 10000}, [1e-3, _alpha_max * 1.01], _data, SEED)
assert _lasso_rows[-1]["n_zero"] == 5, "Lasso above alpha_max must zero every coefficient"
assert _lasso_rows[0]["n_zero"] < 5, "Lasso with a tiny alpha must keep coefficients"

_grid_rows = [{"alpha": 0.1, "rmse_val": 5.0}, {"alpha": 1.0, "rmse_val": 3.0},
              {"alpha": 10.0, "rmse_val": 3.0}, {"alpha": 100.0, "rmse_val": 9.0}]
_best = pick_best_alpha(_grid_rows)
assert _best["alpha"] == 1.0 and _best["at_border"] is False, "pick_best_alpha: exact tie goes to the smaller alpha"
_border = pick_best_alpha([{"alpha": 0.1, "rmse_val": 1.0}, {"alpha": 1.0, "rmse_val": 2.0}])
assert _border["alpha"] == 0.1 and _border["at_border"] is True, "pick_best_alpha: border must be flagged"

# ---- S5: tables, formatting, simplicity, ranking ----
_tab = make_table([[1, 2], [3, 4]], ["a", "b"])
assert list(_tab.columns) == ["a", "b"] and list(_tab.index) == [1, 2], "make_table: columns and 1-based index"
assert format_config({"C": 1.0, "max_iter": 1000}) == "C=1.0, max_iter=1000", "format_config"
assert format_config({}) == "sin hiperparámetros fijados", "format_config: empty config"
assert format_warnings([]) == "—", "format_warnings: empty"
assert format_warnings([{"category": "ConvergenceWarning", "message": "x"}] * 2) == "ConvergenceWarning ×2"
assert simplicity_score(FrozenModel("m", {"a": 1, "b": 2}, None, 3)) == 5, "simplicity_score"
assert rank_models({"a": 0.9, "b": 0.9, "c": 0.8}, True) == {"a": 1, "b": 1, "c": 3}, "rank_models: ties"
assert rank_models({"a": 2.0, "b": 1.0}, False) == {"a": 2, "b": 1}, "rank_models: lower is better"

# ---- S5: indistinguishable_*, degenerate, top errors ----
assert indistinguishable_cls({"a": 2, "b": 3, "c": 4}, "a") == ["b"], "cls: 1 error apart included, 2 excluded"
assert indistinguishable_cls({"a": 2, "b": 1}, "a") == ["b"], "cls: fewer errors than the best is included"
assert indistinguishable_reg({"a": 100.0, "b": 105.0, "c": 105.01}, "a") == ["b"], "reg: exactly 5% included"
assert never_predicted_classes([0, 0, 1], [0, 1, 2]) == [2], "never_predicted_classes"
assert is_degenerate_cls([0, 0, 1], [0, 1, 2]) and not is_degenerate_cls([0, 1, 2], [0, 1, 2]), "is_degenerate_cls"
_zero, _ = fit_capturing_warnings(build_regressor("lasso", {"alpha": 1e6, "max_iter": 1000}, SEED), _Xa, _ya)
_ridge, _ = fit_capturing_warnings(build_regressor("ridge", {"alpha": 1.0}, SEED), _Xa, _ya)
assert is_degenerate_reg(_zero) and not is_degenerate_reg(_ridge), "is_degenerate_reg"
assert top_abs_errors([0, 0, 0, 0], [1, -3, 3, 0.5], k=2).tolist() == [1, 2], "top_abs_errors: ties, smaller position"

# ---- S5: pick_two_errors (three scenarios plus a validation-only case) ----
_order = ["m1", "m2", "m3"]
_yt = np.array([0, 1, 2, 0, 1, 2])
_it = np.array([10, 11, 12, 13, 14, 15])
_pt = {"m1": np.array([0, 2, 2, 0, 0, 2]), "m2": np.array([0, 2, 2, 0, 1, 2]), "m3": np.array([0, 1, 0, 0, 0, 2])}
_yv = np.array([0, 1, 2, 0])
_iv = np.array([20, 21, 22, 23])
_pv = {"m1": np.array([0, 2, 2, 0]), "m2": np.array([0, 1, 2, 0]), "m3": np.array([0, 2, 2, 1])}
_two = pick_two_errors(_pt, _pv, _yt, _yv, _order, _it, _iv)
assert [e["index"] for e in _two] == [11, 14] and all(e["partition"] == "prueba" for e in _two), \
    "pick_two_errors: test errors ordered by number of failing models"
assert _two[0]["model"] == "m1" and _two[0]["also_failed"] == 1, "pick_two_errors: first failing model and count"
_one_test = {"m1": np.array([0, 1, 2, 0, 1, 2]), "m2": np.array([0, 2, 2, 0, 1, 2]), "m3": _yt.copy()}
_mixed = pick_two_errors(_one_test, _pv, _yt, _yv, _order, _it, _iv)
assert [e["partition"] for e in _mixed] == ["prueba", "validación"], "pick_two_errors: test first, then validation"
assert _mixed[1]["model"] == "m3" and _mixed[1]["index"] == 21, \
    "pick_two_errors: validation error comes from the model with most validation errors"
assert "optimista" in _mixed[1]["note"], "pick_two_errors: optimistic-bias note"
_clean = {m: _yt.copy() for m in _order}
_clean_v = {m: _yv.copy() for m in _order}
assert pick_two_errors(_clean, _clean_v, _yt, _yv, _order, _it, _iv) == [], "pick_two_errors: no errors anywhere"
_val_only = pick_two_errors(_clean, _pv, _yt, _yv, _order, _it, _iv)
assert [e["index"] for e in _val_only] == [21, 23] and {e["model"] for e in _val_only} == {"m3"}, \
    "pick_two_errors: validation-only errors come from the model with most validation errors"

# ---- S5: result_digest ----
_fp = result_digest(np.array([0.5, 1.5]), np.array([[1.0, 2.0]]))
assert _fp == result_digest(np.array([0.5, 1.5]), np.array([[1.0, 2.0]])), "fingerprint: same input"
assert _fp != result_digest(np.array([0.5, 1.6]), np.array([[1.0, 2.0]])), "fingerprint: altered input"
assert _fp == result_digest(np.array([0.5 + 1e-9, 1.5]), np.array([[1.0, 2.0]])), "fingerprint: 6 decimals"

# 1. Datos y protocolo experimental

## Dataset inicial: Iris

Carga el dataset Iris, incluido en scikit-learn.

Identifica:

- Qué representa cada instancia.
- Número de instancias y características.
- Nombre y significado de cada característica.
- Clases y cantidad de ejemplos por clase.
- Dimensiones de la matriz de características y del vector de etiquetas.

Presenta una tabla con cinco ejemplos y un gráfico de distribución de clases.


## E1.P (pendiente)

In [7]:
# [E1.C] (pending)

## E1.I (pendiente)

# 2. Comparación de clasificadores

## Predicciones iniciales

Antes de implementar, responde:

1. ¿Qué modelos crees que serán sensibles a las escalas de las características?
2. ¿Qué modelo esperas que produzca fronteras de decisión más simples?
3. ¿Por qué no podemos asegurar de antemano cuál será el mejor?

## Modelos

Construye un modelo de clasificación para la clase y compara:

- Gaussian Naive Bayes.
- Regresión logística multiclase.
- SVM lineal.
- Árbol de decisión.
- Random Forest.
- Gradient Boosting.
- MLP de clasificación.

Usa características estandarizadas para los clasificadores.

## Procedimiento

1. Define configuraciones iniciales.
2. Entrena con la partición de entrenamiento.
3. Compara accuracy y F1 macro en validación.
4. Realiza ajustes limitados usando solamente validación.
5. Congela las configuraciones.
6. Evalúa todos los modelos finales sobre la misma partición de prueba.

La tabla de prueba describe sus resultados; no debe iniciar una nueva ronda de ajuste.

## Resultados

Presenta una tabla con:

| Modelo | Transformación | Hiperparámetros principales | Accuracy de prueba | F1 macro de prueba |
|---|---|---|---|---|

Incluye matrices de confusión de Naive Bayes, SVM y MLP.

## Interpretación

1. ¿Cuál fue el modelo con el mejor desempeño?
2. ¿Cuál fue el modelo más sencillo de programar?
3. ¿Accuracy y F1 macro cuentan la misma historia?
4. ¿Cuál es la diferencia entre las métricas macro y micro?
5. ¿Qué aporta el MLP frente a un modelo lineal?

Selecciona dos errores de clasificación. Examina sus características y explica qué puedes concluir y qué no puedes concluir sobre la causa del error.

## E2.P (pendiente)

In [8]:
# [E2.C] (pending)

## E2.I (pendiente)

# 3. Comparación de modelos de regresión

## Dataset: Diabetes

Carga Diabetes, incluido en scikit-learn.

Describe qué representa el objetivo numérico y qué variables contiene el dataset.

Comprueba en la descripción del dataset si las características ya incluyen transformaciones. No interpretes sus valores automáticamente como mediciones en unidades originales.

## Predicción inicial

¿Esperas que un modelo más flexible produzca siempre un menor error en datos nuevos? Justifica.

## Modelos

Compara:

- Regresión lineal.
- Ridge.
- Lasso.
- kNN de regresión.
- Árbol de regresión.
- Random Forest de regresión.
- Gradient Boosting de regresión.
- MLP de regresión.

Usa escalamiento mediante pipelines cuando corresponda.

Si transformas el objetivo para facilitar el entrenamiento del MLP, ajusta esa transformación solo con entrenamiento y devuelve las predicciones a la escala original antes de evaluarlas.

## Procedimiento

Usa entrenamiento para ajustar y validación para decidir configuraciones. Una vez congeladas, evalúa los modelos sobre prueba.

Presenta:

| Modelo | MAE | RMSE | R² |
|---|---|---|---|

Para regresión lineal y MLP, incluye:

- Gráfico de valores reales frente a predicciones.
- Gráfico de residuos.
- Dos ejemplos con errores absolutos grandes.

## Regularización

Compara varias intensidades de penalización en Ridge y Lasso usando validación.

Examina:

- Magnitud de los coeficientes.
- Cantidad de coeficientes nulos en Lasso.
- Error de entrenamiento y validación.

## Interpretación

1. ¿Qué modelo tuvo el mejor desempeño?
2. ¿Por qué MAE y RMSE pueden favorecer resultados diferentes?
3. ¿Cómo se interpreta un R² negativo?
4. ¿Qué efecto tuvo aumentar la regularización?
5. ¿Qué puede explicar los errores grandes?
6. ¿Un coeficiente predictivo demuestra una relación causal?

## E3.P (pendiente)

In [9]:
# [E3.C] (pending)

## E3.I (pendiente)